1. ⚙️ Setup & Authentication
First, install the Google Ads client library and authenticate your session. **bold text****bold text**

Note: eed google-ads.yaml file containing your Developer Token and Client Secrets.

In [ ]:
# Install the Google Ads API client
!pip install google-ads -q

import os
import pandas as pd
from google.ads.googleads.client import GoogleAdsClient
from google.colab import files

# Upload your google-ads.yaml file here
uploaded = files.upload()
client = GoogleAdsClient.load_from_storage("google-ads.yaml")
CUSTOMER_ID = 'YOUR_GOOGLE_ADS_ID' # e.g. '1234567890'

2. 📊 East District Capacity Data
Define baseline for the East districts. In a production environment, you would replace this dictionary with a pd.read_sql or a Google Sheets import.

In [ ]:
# District Data for the East Region
east_districts = [
    {"name": "Norwich", "geo_id": "1007011", "current": 42, "max": 50},
    {"name": "Ipswich", "geo_id": "1006884", "current": 15, "max": 60},
    {"name": "Cambridge", "geo_id": "1006524", "current": 58, "max": 60},
    {"name": "Peterborough", "geo_id": "1007038", "current": 25, "max": 100},
]

df = pd.DataFrame(east_districts)
df['utilisation'] = df['current'] / df['max']

Logic: True Capacity Multiplier
....

In [ ]:
def calculate_multiplier(u):
    if u < 0.40: return 1.5   # Under-utilised: Bid Aggressively
    if u < 0.75: return 1.0   # Optimal: Steady
    if u < 0.90: return 0.5   # Nearing Capacity: Taper off
    return 0.1                # Full: Minimal Bid

df['multiplier'] = df['utilisation'].apply(calculate_multiplier)

# Display the calculation results
print("Target Multipliers for East Region:")
df[['name', 'utilisation', 'multiplier']]

API Push: Mutate Conversion Value Rules
This block iterates through your East districts and updates the Google Ads rules. This ensures the AI optimizes for volume where you actually have space.

In [ ]:
def update_conversion_value_rule(client, customer_id, geo_id, multiplier):
    rule_service = client.get_service("ConversionValueRuleService")
    operation = client.get_type("ConversionValueRuleOperation")

    # Define Rule Properties
    rule = operation.create
    rule.action.operation = client.enums.ValueRuleActionOperationEnum.MULTIPLY
    rule.action.value = multiplier

    # Set Geo Condition
    geo_constant = client.get_service("GeoTargetConstantService").geo_target_constant_path(geo_id)
    rule.geo_location_condition.geo_target_constants.append(geo_constant)

    # Execute API Call
    try:
        response = rule_service.mutate_conversion_value_rules(customer_id=customer_id, operations=[operation])
        return response.results[0].resource_name
    except Exception as e:
        return f"Error: {e}"

# Run updates for all districts in the dataframe
df['api_response'] = df.apply(lambda x: update_conversion_value_rule(client, CUSTOMER_ID, x['geo_id'], x['multiplier']), axis=1)
print("Update Complete.")

## **Step 2: Setup & Capacity Logic**

We will use the logic defined in the [Google API Documentation](https://developers.google.com/google-ads/api/docs/conversions/conversion-value-rules#:~:text=Conversion%20value%20rules%20allow%20you,using%20the%20Google%20Ads%20API.) to create **"Mutate"** operations.



In this step, the code calculates the required adjustment for each district in the **East region**. By converting your **True Available Capacity** into a numerical multiplier, we create a set of instructions that the API can use to update your campaign's bidding behavior in real-time.

### **API Call Instructions:**

To implement these changes, you will interact with the **`ConversionValueRuleService`** using the following workflow:

1.  **Initialize the Service:** Use the `GoogleAdsClient` to access the `ConversionValueRuleService`.
2.  **Define the Operation:** Create a `ConversionValueRuleOperation` to specify whether you are creating a new rule or updating an existing one.
3.  **Set the Action (The "Then"):** Specify the `operation` as `MULTIPLY` and pass the calculated `multiplier` (e.g., `0.2` for a district at capacity).
4.  **Set the Condition (The "If"):** Attach the **Geo Target Constant ID** for the specific district (e.g., Norwich or Ipswich) to the `geo_location_condition`.
5.  **Execute Mutate:** Call `mutate_conversion_value_rules` to push the changes live to your Google Ads account.

In [ ]:
developer_token: INSERT_TOKEN_HERE
client_id: INSERT_CLIENT_ID_HERE
client_secret: INSERT_CLIENT_SECRET_HERE
refresh_token: INSERT_REFRESH_TOKEN_HERE
login_customer_id: INSERT_MCC_ID_HERE

In [ ]:
!pip install google-ads -q

import pandas as pd
from google.ads.googleads.client import GoogleAdsClient

# Load credentials from the yaml file you uploaded
client = GoogleAdsClient.load_from_storage("google-ads.yaml")
CUSTOMER_ID = '1234567890' # Your specific account ID

# 1. Define East Districts and their "True Capacity"
# Note: Using Criteria IDs for precision (Norwich, Ipswich, Cambridge)
data = [
    {"name": "Norwich", "geo_id": "1007011", "util": 0.35}, # High Capacity (35% full)
    {"name": "Ipswich", "geo_id": "1006884", "util": 0.85}, # Optimal (85% full)
    {"name": "Cambridge", "geo_id": "1006524", "util": 0.98} # FULL (98% full)
]

df = pd.DataFrame(data)

# 2. Calculate the Multiplier
def get_multiplier(u):
    if u < 0.40: return 1.5  # Boost value to fill empty slots
    if u > 0.90: return 0.2  # Slash value to stop the AI from over-spending
    return 1.0               # Maintain current CPA

df['multiplier'] = df['util'].apply(get_multiplier)

**Step 3: Call the API (The "Mutate" Operation)**
This code creates or updates the Conversion Value Rules specifically for your East region locations.

In [ ]:
def update_value_rule(client, customer_id, geo_id, multiplier):
    # Get the service
    rule_service = client.get_service("ConversionValueRuleService")

    # Create the operation
    operation = client.get_type("ConversionValueRuleOperation")
    rule = operation.create

    # Set the Action: MULTIPLY the conversion value
    rule.action.operation = client.enums.ValueRuleActionOperationEnum.MULTIPLY
    rule.action.value = multiplier

    # Set the Condition: Target the specific District ID
    geo_path = client.get_service("GeoTargetConstantService").geo_target_constant_path(geo_id)
    rule.geo_location_condition.geo_target_constants.append(geo_path)

    # Execute the API call
    response = rule_service.mutate_conversion_value_rules(
        customer_id=customer_id,
        operations=[operation]
    )
    return response.results[0].resource_name

# Apply to your East District Dataframe
df['rule_resource_name'] = df.apply(
    lambda x: update_value_rule(client, CUSTOMER_ID, x['geo_id'], x['multiplier']), axis=1
)

print("Rules Updated Successfully in the East Region.")
df[['name', 'util', 'multiplier', 'rule_resource_name']]

**💡 Instructions for calling the API effectively:**
The "Set" vs. the "Rule": Creating the ConversionValueRule (as shown above) defines the math. To make it work, you must then create a ConversionValueRuleSet and link it to your East Regional Campaign ID.

**API Limits:** You can update rules multiple times a day, but for a regional test, once every 24 hours (morning run) is usually sufficient for the Smart Bidding algorithm to stay stable.

**Validation: **After running the script, go to Goals > Conversions > Value Rules in the Google Ads UI. You should see your districts (Norwich, Cambridge, etc.) listed with their new multipliers.

**Monitoring:** Use the Value / Conv. column in your campaign report. If the value in Cambridge drops significantly, you know the API is successfully telling the AI to stop bidding high there.

In [ ]:
def link_rules_to_campaign(client, customer_id, campaign_id, rule_resource_names):
    """
    Creates a ConversionValueRuleSet to apply specific rules to a campaign.
    """
    # 1. Initialize the Rule Set Service
    rule_set_service = client.get_service("ConversionValueRuleSetService")
    operation = client.get_type("ConversionValueRuleSetOperation")

    rule_set = operation.create

    # 2. Attach the rules we just created
    # rule_resource_names is a list of strings from your previous API responses
    rule_set.conversion_value_rules.extend(rule_resource_names)

    # 3. Link it to your East Regional Campaign
    campaign_service = client.get_service("CampaignService")
    rule_set.campaign = campaign_service.campaign_path(customer_id, campaign_id)

    # 4. Define the dimensions (In this case, Geo Location)
    rule_set.dimensions.append(client.enums.ValueRuleSetDimensionEnum.GEO_LOCATION)

    # 5. Push to Google Ads
    try:
        response = rule_set_service.mutate_conversion_value_rule_sets(
            customer_id=customer_id,
            operations=[operation]
        )
        print(f"Success! Rule Set created: {response.results[0].resource_name}")
        return response.results[0].resource_name
    except Exception as e:
        print(f"Error linking rules: {e}")
        return None

# --- EXECUTION ---
# Replace 'YOUR_CAMPAIGN_ID' with the ID of your new consolidated East campaign
EAST_CAMPAIGN_ID = '987654321'
rule_list = df['rule_resource_name'].tolist()

link_rules_to_campaign(client, CUSTOMER_ID, EAST_CAMPAIGN_ID, rule_list)

**Step 4: Weather Signals & The Profitability Feedback Loop**
We now integrate external demand triggers and internal performance data to create a "Master Multiplier."

**Weather Demand (OpenWeather API):** We call the OpenWeather Technology to check for "High Demand" conditions (e.g., storms or extreme temperatures) in each East district. If the weather matches your business's peak demand profile, we boost the value rule.

**Profitability Feedback Loop: **The script queries the Google Ads API for the actual Cost per Conversion (CPA) of each district.

If a district has low utilisation BUT the CPA is too high, the script will override the "push" and lower the multiplier to protect your margins.

If CPA is low, it signals "Green Light" to spend more and hit that 85% utilisation target faster.

In [ ]:
import requests

# --- CONFIGURATION ---
OPENWEATHER_API_KEY = "YOUR_OPENWEATHER_KEY"
TARGET_CPA = 9.00  # Based on your initial data (£8.99 avg)

# 1. Fetch Weather Signal
def get_weather_multiplier(city_name):
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city_name},GB&appid={OPENWEATHER_API_KEY}"
    try:
        data = requests.get(url).json()
        condition = data['weather'][0]['main']
        # Example Logic: If it's Rain/Stormy, demand increases for our service
        if condition in ['Rain', 'Thunderstorm', 'Drizzle']:
            return 1.3
        return 1.0
    except:
        return 1.0

# 2. Fetch Performance Data (Google Ads API Query)
def get_recent_cpa(client, customer_id, geo_id):
    ga_service = client.get_service("GoogleAdsService")
    # Query for the last 7 days for that specific location
    query = f"""
        SELECT metrics.cost_per_conversion
        FROM location_view
        WHERE segments.date DURING LAST_7_DAYS
        AND metrics.conversions > 0
        AND campaign.id = 'YOUR_EAST_CAMPAIGN_ID'
    """
    # Note: Simplified for example. In production, you'd filter by criteria_id.
    return 8.50 # Mocking a return value of £8.50

# 3. The "Master" Multiplier Calculation
def calculate_master_multiplier(row):
    # Start with base capacity multiplier
    u = row['util']
    if u < 0.40: m = 1.5
    elif u > 0.90: m = 0.2
    else: m = 1.0

    # Apply Weather Layer
    weather_m = get_weather_multiplier(row['name'])
    m *= weather_m

    # Apply Performance Feedback Layer
    actual_cpa = get_recent_cpa(client, CUSTOMER_ID, row['geo_id'])
    if actual_cpa < TARGET_CPA * 0.8: # Very profitable
        m *= 1.2
    elif actual_cpa > TARGET_CPA * 1.2: # Too expensive
        m *= 0.7

    return round(m, 2)

# --- EXECUTION ---
df['final_multiplier'] = df.apply(calculate_master_multiplier, axis=1)

print("Final Strategic Multipliers (Capacity + Weather + Performance):")
print(df[['name', 'util', 'final_multiplier']])

**The "Infinite Loop" Control Center**
Copy this block into your final Colab cell. This code is designed to run in a loop (e.g., every hour or daily) to keep your rules perfectly synced with the real world.

In [ ]:
import requests
import time
from datetime import datetime

# --- SETTINGS ---
OPENWEATHER_API_KEY = "YOUR_API_KEY"
TARGET_CPA = 8.50 # Based on East's current performance
LOOP_INTERVAL_SECONDS = 3600 # Run every hour

def run_strategic_update():
    print(f"--- Starting Sync: {datetime.now()} ---")

    # 1. Pull current performance data via Google Ads Query
    # Logic: Get CPA per location for the last 7 days
    performance_df = get_performance_data(client, CUSTOMER_ID)

    # 2. Iterate through districts to find the 'Master Multiplier'
    rule_ids_to_update = []

    for index, row in df.iterrows():
        # A. Base Capacity Multiplier
        m = get_capacity_multiplier(row['util'])

        # B. Weather Signal
        weather_condition = get_weather(row['name'], OPENWEATHER_API_KEY)
        if weather_condition in ['Rain', 'Snow']: m *= 1.3

        # C. Feedback Loop: CPA Adjustment
        actual_cpa = performance_df.loc[row['name'], 'cpa']
        if actual_cpa > TARGET_CPA * 1.15: # CPA is 15% too high
            m *= 0.8
        elif actual_cpa < TARGET_CPA * 0.85: # CPA is 15% lower than target
            m *= 1.2

        # 3. Push update to Google Ads API
        new_multiplier = round(m, 2)
        update_conversion_value_rule(client, CUSTOMER_ID, row['geo_id'], new_multiplier)

    print("Dashboard Updated: All rules synced with Capacity + Weather + CPA.")

# Start the Loop
while True:
    run_strategic_update()
    time.sleep(LOOP_INTERVAL_SECONDS)